In [1]:
import pandas as pd
import numpy as np

In [2]:
STRING_INFO = "../data/raw/9606.protein.info.v12.0.csv"
STRING_LINKS = "../data/raw/9606.protein.links.detailed.v12.0.csv"

SAVE_EDGES = "../data/processed/string_graph_edges.csv"
SAVE_NODES = "../data/processed/string_nodes.csv"

Load STRING Protein Information

In [3]:
protein_info = pd.read_csv(STRING_INFO)

print("Protein info shape:", protein_info.shape)

protein_info.head()

Protein info shape: (19699, 4)


,#string_protein_id,preferred_name,protein_size,annotation
0,9606.ENSP00000000233,ARF5,180,ADP-ribosylation factor 5; GTP-binding protein...
1,9606.ENSP00000000412,M6PR,277,Cation-dependent mannose-6-phosphate receptor;...
2,9606.ENSP00000001008,FKBP4,459,"Peptidyl-prolyl cis-trans isomerase FKBP4, N-t..."
3,9606.ENSP00000001146,CYP26B1,512,Cytochrome P450 26B1; Involved in the metaboli...
4,9606.ENSP00000002125,NDUFAF7,441,"Protein arginine methyltransferase NDUFAF7, mi..."


Clean Column Names

In [4]:
protein_info.columns = protein_info.columns.str.strip()

print(protein_info.columns)

Index(['#string_protein_id', 'preferred_name', 'protein_size', 'annotation'], dtype='object')


Create Protein → Gene Mapping

In [5]:
protein_to_gene = dict(
    zip(
        protein_info["#string_protein_id"],
        protein_info["preferred_name"]
    )
)

print("Total protein mappings:", len(protein_to_gene))

Total protein mappings: 19699


Load STRING Interaction Links

In [6]:
links = pd.read_csv(
    STRING_LINKS,
    sep=" "
)

print("Links shape:", links.shape)
links.head()

Links shape: (13715404, 10)


,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
0,9606.ENSP00000000233,9606.ENSP00000356607,0,0,0,45,134,0,81,173
1,9606.ENSP00000000233,9606.ENSP00000427567,0,0,0,0,128,0,70,154
2,9606.ENSP00000000233,9606.ENSP00000253413,0,0,0,118,49,0,69,151
3,9606.ENSP00000000233,9606.ENSP00000493357,0,0,0,56,53,0,457,471
4,9606.ENSP00000000233,9606.ENSP00000324127,0,0,0,0,46,0,197,201


Filter High Confidence Interactions

In [7]:
links = links[links["combined_score"] > 700]

print("Filtered interactions:", len(links))

Filtered interactions: 472000


Map Protein IDs → Gene Symbols

In [8]:
links["gene1"] = links["protein1"].map(protein_to_gene)
links["gene2"] = links["protein2"].map(protein_to_gene)

Remove Unmapped Proteins

In [9]:
links = links.dropna(subset=["gene1", "gene2"])

print("Interactions after mapping:", len(links))

Interactions after mapping: 472000


Create Gene Interaction Edges

In [10]:
edges = links[["gene1", "gene2"]].drop_duplicates()

edges.head()

,gene1,gene2
85,ARF5,ACAP1
130,ARF5,COPA
160,ARF5,RAB11FIP3
197,ARF5,COPB2
268,ARF5,COPE


Save Graph Edges

In [11]:
edges.to_csv(SAVE_EDGES, index=False)

print("Saved edges to:", SAVE_EDGES)

Saved edges to: ../data/processed/string_graph_edges.csv


Create Node List

In [12]:
nodes = pd.unique(edges[["gene1", "gene2"]].values.ravel())

nodes = pd.DataFrame(nodes, columns=["gene"])

nodes.head()

,gene
0,ARF5
1,ACAP1
2,COPA
3,RAB11FIP3
4,COPB2


Save Node List

In [13]:
nodes.to_csv(SAVE_NODES, index=False)

print("Saved nodes to:", SAVE_NODES)

Saved nodes to: ../data/processed/string_nodes.csv


Graph Statistics

In [14]:
print("Total Nodes:", len(nodes))
print("Total Edges:", len(edges))

Total Nodes: 16185
Total Edges: 472000
